# ICU Length of Stay Prediction

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl
from datetime import timedelta

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics_regression, preprocess_inference_results
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

/home/henri/ethos-benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.length_of_stay import LosPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = LosPredictionDataset(path, offset=timedelta(hours=24))

Found 8637 hospital stays with an outcome token.


# MIMIC

In [9]:
results_dir = PROJECT_ROOT / "debug_results"
los_pred_result_dir = results_dir / Task.LOS_PREDICTION

def preprocess_los_prediction_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual") == ST.ICU_DISCHARGE,
        expected_expr=pl.col("expected") == ST.ICU_DISCHARGE,
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

In [50]:
los_pred_results = []
for result_fp in los_pred_result_dir.iterdir():
    df = preprocess_los_prediction_results(result_fp)
    rep_num = df["counts"].mean()
    res_full = compute_and_print_metrics_regression(df["true_token_time"], df["token_time"], counts=df["counts"][0])

2026-10-19 12:38:47.100 | WARNING  | ethos.metrics:preprocess_inference_results:320 - Dropped 3 (1.36%) ambiguous results.


MAE: 9 days, 1:42:51.812387 for 220 samples
